In [25]:
import pickle
import matplotlib.pyplot as plt
import numpy as np
import cvxpy as cp
import os
from google.colab import drive
import warnings

# Tắt tất cả các warning liên quan đến user (bao gồm cái của cvxpy)
warnings.filterwarnings("ignore", category=UserWarning)

# HOẶC: Chỉ tắt đúng cái warning về ma trận này (An toàn hơn)
warnings.filterwarnings("ignore", message=".*matrix multiplication.*")

# ==========================================
# 1. MOUNT GOOGLE DRIVE
# ==========================================
# Bước này sẽ yêu cầu bạn cấp quyền truy cập Drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [26]:
# --- CES / Linear (m = CES exponent, m=1 → Linear, m='inf' → Leontief-like max) ---
def calculate_ces_utility(allocation_vec, valuation_vec, m_ces = 1/2):
    eps = 1e-9
    return np.power(np.power(allocation_vec + eps, m_ces).T @ valuation_vec, (1/m_ces))


# --- COBB–DOUGLAS UTILITY ---
def calculate_cd_utility(allocation_vec, valuations_vec):
    eps = 1e-12
    v = np.atleast_2d(valuations_vec)
    x = np.atleast_2d(allocation_vec)

    # Normalize weights α_j = v_j / sum(v)
    weights = v / (np.sum(v, axis=1, keepdims=True))

    # log utility = Σ α_j log(x_j)
    log_u = np.sum(weights * np.log(x + eps), axis=1)

    u = np.exp(log_u)

    return u


# --- LEONTIEF UTILITY ---
def calculate_leo_utility(allocation_vec, valuations_vec):
    v = np.atleast_2d(valuations_vec)
    x = np.atleast_2d(allocation_vec)
    # 1. Khởi tạo mảng kết quả là Vô cực (Infinity)
    # Để nếu v=0, giá trị tại đó vẫn là Inf và không bị hàm min chọn phải
    ratios = np.full_like(x, 1e30)

    # 2. Chỉ thực hiện phép chia ở những nơi v > 0
    # Ghi đè kết quả phép chia vào mảng ratios
    np.divide(x, v, out=ratios, where=(v > 1e-12))

    # 3. Lấy Min theo hàng (axis=1)
    u = np.min(ratios, axis=1)

    return u


## 1a

In [32]:
import numpy as np
import matplotlib.pyplot as plt
import cvxpy as cp
import os
import time

# ==============================================================================
# 1. CORE FUNCTIONS
# ==============================================================================

def build_A_matrix(n_goods, n_slots):
    """Tạo ma trận ánh xạ Goods -> Slots"""
    A = np.zeros((n_slots, n_goods))
    for j in range(n_goods):
        A[j % n_slots, j] = 1.0
    return A

import numpy as np

def buyer_best_response_cvx(v_i, p, A, q_i, B_i, utility_type="Linear"):
    """
    Phiên bản siêu tốc của buyer_best_response.
    Sử dụng Closed-form solution cho mọi hàm utility.
    KHÔNG DÙNG CVXPY.
    """
    # 1. Tính giá hiệu dụng (Effective Price)
    # p: (M,), A.T @ q_i: (M,)
    effective_price = p + (A.T @ q_i)

    # An toàn: Đảm bảo giá > 0 để tránh chia cho 0
    safe_price = np.maximum(effective_price, 1e-9)

    m_goods = len(v_i)
    x = np.zeros(m_goods)

    # =========================================================
    # 1. LINEAR UTILITY: U = sum(v * x)
    # =========================================================
    if utility_type == "Linear":
        # Chiến thuật: Bang-per-buck (Mua tất tay món hời nhất)
        bang_per_buck = v_i / safe_price
        best_idx = np.argmax(bang_per_buck)
        x[best_idx] = B_i / safe_price[best_idx]

    # =========================================================
    # 2. COBB-DOUGLAS: U = prod(x^alpha) hoặc sum(alpha * log(x))
    # =========================================================
    elif utility_type == "Cobb-Douglas":
        # Chuẩn hóa alpha (Bắt buộc để thỏa mãn Budget constraint)
        sum_v = np.sum(v_i)
        if sum_v > 0:
            alpha = v_i / sum_v
        else:
            alpha = np.zeros_like(v_i) # Tránh lỗi nếu v toàn 0

        # Công thức: Chi tiêu đúng tỷ lệ alpha
        # x_i = (alpha_i * Budget) / Price_i
        x = (alpha * B_i) / safe_price

    # =========================================================
    # 3. LEONTIEF: U = min(x / v)
    # =========================================================
    elif utility_type == "Leontief":
        # Chiến thuật: Mua theo tỷ lệ cố định của v
        # Giá của 1 combo chuẩn = sum(v_i * p_i)
        cost_of_one_bundle = np.dot(v_i, safe_price)

        if cost_of_one_bundle > 0:
            # Số lượng combo mua được
            num_bundles = B_i / cost_of_one_bundle
            x = num_bundles * v_i
        else:
            x = np.zeros(m_goods)

    # =========================================================
    # 4. CES UTILITY: U = (sum v * x^rho)^(1/rho)
    # =========================================================
    elif utility_type == "CES_8":
        # CẤU HÌNH rho (m) TẠI ĐÂY
        # Trong code cũ bạn để m = 1/2
        rho = 0.5

        # --- CASE A: CONCAVE (rho < 1, rho != 0) ---
        # Đây là trường hợp thay thế (Substitute) -> Mua nhiều loại
        if rho < 1:
            # Tính Sigma (Elasticity of Substitution)
            # sigma = 1 / (1 - rho)
            sigma = 1.0 / (1.0 - rho)

            # Tính phần tử tỷ lệ: Term_i = (v_i / p_i)^sigma
            # Dùng np.maximum cho v_i để tránh v=0 gây lỗi log hoặc mũ âm
            term = np.power(v_i / safe_price, sigma)

            # Tính mẫu số chung: Sum (p_j * term_j)
            denom = np.dot(safe_price, term)

            if denom > 0:
                # x_i = (B * term_i) / denom
                x = (B_i * term) / denom
            else:
                 # Fallback nếu v=0 hết
                 x = np.zeros(m_goods)

        # --- CASE B: CONVEX (rho > 1) ---
        # Đây là trường hợp "Winner Takes All" giống Linear
        else:
            # So sánh tỷ lệ: v^(1/rho) / p
            v_transformed = np.power(v_i, 1.0/rho)
            bang_per_buck = v_transformed / safe_price

            best_idx = np.argmax(bang_per_buck)
            x[best_idx] = B_i / safe_price[best_idx]

    return x

def solve_centralized_optimal_fair(valuations, budgets, supply_s, b_mat, A, utility_type="Linear"):
    """
    Returns:
        optimal_value (float): Giá trị hàm mục tiêu tối ưu.
        optimal_X (np.ndarray): Ma trận phân bổ tối ưu (N x M).
    """
    n, m = valuations.shape
    X = cp.Variable((n, m), nonneg=True)

    constraints = [
        cp.sum(X, axis=0) <= supply_s
    ]
    for i in range(n):
        constraints.append(A @ X[i] <= b_mat[i])

    # --- XÂY DỰNG OBJECTIVE ---
    if utility_type == "Linear":
        utilities = cp.sum(cp.multiply(valuations, X), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "Cobb-Douglas":
        eps = 1e-12
        alpha = valuations / (np.sum(valuations, axis=1, keepdims=True))
        log_utilities = cp.sum(cp.multiply(alpha, cp.log(X + eps)), axis=1)
        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    elif utility_type == "Leontief":
        # Lưu ý: Leontief trong CVXPY có thể phức tạp/chậm với quy mô lớn
        inv_valuations = np.divide(
            1.0,
            valuations,
            out=np.full_like(valuations, 1e30),
            where=(valuations > 0)
        )

        # 2. Nhân X với ma trận nghịch đảo hằng số này
        # ratio_matrix[i, j] = X[i, j] * (1 / v[i, j])
        weighted_X = cp.multiply(X, inv_valuations)

        # 3. Lấy min theo hàng
        utilities = cp.min(weighted_X, axis=1)

        # 4. Tính hàm mục tiêu
        primal_utility = cp.sum(cp.multiply(budgets, cp.log(utilities + 1e-12)))

    elif utility_type == "CES_8":
        # Nếu chạy Solver tập trung, m NÊN nhỏ hơn hoặc bằng 1 (ví dụ 0.5 hoặc -1).
        # Nếu đặt m = 8, Solver ECOS/SCS sẽ KHÔNG giải được (báo lỗi DCPError).
        eps = 1e-9
        m_ces = 1/2

        # Công thức: log_util = (1/m) * log( sum( v * x^m ) )
        # Tính tổng trọng số lũy thừa theo hàng (axis=1)
        inner_term = cp.sum(cp.multiply(valuations, cp.power(X + eps, m_ces)), axis=1)

        # Logarit hóa hàm mục tiêu
        log_utilities = (1.0 / m_ces) * cp.log(inner_term)

        primal_utility = cp.sum(cp.multiply(budgets, log_utilities))

    objective = cp.Maximize(primal_utility)
    prob = cp.Problem(objective, constraints)

    print(f"--- Solving Centralized Fair Problem ({utility_type}) ---")
    try:
        prob.solve(solver=cp.ECOS, verbose=False)
    except:
        prob.solve(solver=cp.SCS, verbose=False)

    # TRẢ VỀ: (Giá trị mục tiêu, Ma trận X tối ưu)
    return prob.value, X.value


def one_sample_sgd_fastlog(
    A: np.ndarray, b: np.ndarray, supply_s: np.ndarray, q0: np.ndarray,
    valuations: np.ndarray, budgets: np.ndarray, p0: np.ndarray,
    lr_p=0.01, lr_q=0.01, num_iters=1000, seed=42, log_freq=10,
    utility_type="Linear"
):
    rng = np.random.default_rng(seed)
    n, m = valuations.shape

    # Copy biến dual để cập nhật
    p = p0.astype(np.float64).copy()
    q = q0.astype(np.float64).copy()

    # Khởi tạo lịch sử
    obj_hist = []
    real_allocation_hist = []

    # --- BƯỚC 1: KHỞI TẠO (WARM START TẠI T=0) ---
    # Tính toán X ban đầu cho TẤT CẢ người dùng dựa trên p0, q0
    X = np.zeros((n, m))
    util = np.zeros(n)

    for j in range(n):
        # Dùng hàm Analytical siêu tốc
        X[j] = buyer_best_response_cvx(valuations[j], p, A, q[j], budgets[j], utility_type)

        # Tính Utility tương ứng
        if utility_type == "Linear":
            util[j] = valuations[j] @ X[j]
        elif utility_type == "Cobb-Douglas":
            util[j] = calculate_cd_utility(X[j], valuations[j])
        elif utility_type == "Leontief":
            util[j] = calculate_leo_utility(X[j], valuations[j])
        elif utility_type == "CES_8":
            util[j] = calculate_ces_utility(X[j], valuations[j], m_ces=0.5)

    # Tính Tổng cầu ban đầu (D)
    D = X.sum(axis=0)

    # --- BƯỚC 2: LƯU TRẠNG THÁI T=0 (ĐÚNG YÊU CẦU CỦA BẠN) ---
    # Lưu ngay lập tức X0 và Obj0 trước khi vòng lặp SGD bắt đầu

    # Tính Objective ban đầu
    term_p = np.sum(p * supply_s)
    term_q = np.sum(q * b)
    term_u = np.sum(budgets * np.log(util + 1e-12))
    obj_0 = term_p + term_q + term_u - np.sum(budgets) # Trừ hằng số budget (optional)

    # Append vào list
    obj_hist.append(obj_0)
    real_allocation_hist.append(X.copy()) # Lưu bản sao X0

    print(f"--- Start SGD ({num_iters} iterations) | Initial Obj: {obj_0:.2f} ---")
    t0 = time.perf_counter()

    # --- BƯỚC 3: VÒNG LẶP SGD (TỪ T=1 ĐẾN NUM_ITERS) ---
    for t in range(1, num_iters + 1):
        # A. Chọn ngẫu nhiên 1 user
        i = rng.integers(0, n)

        # B. Cập nhật Demand (Incremental)
        D -= X[i]

        # C. Tìm Best Response (Analytical)
        x_new = buyer_best_response_cvx(valuations[i], p, A, q[i], budgets[i], utility_type)

        # Cập nhật X và Utility
        X[i] = x_new

        if utility_type == "Linear":
            util[i] = valuations[i] @ X[i]
        elif utility_type == "Cobb-Douglas":
            util[i] = calculate_cd_utility(X[i], valuations[i])
        elif utility_type == "Leontief":
            util[i] = calculate_leo_utility(X[i], valuations[i])
        elif utility_type == "CES_8":
            util[i] = calculate_ces_utility(X[i], valuations[i], m_ces=0.5)

        D += x_new

        # D. Tính Gradient & Cập nhật Dual
        g_p = D - supply_s
        load_i = A @ x_new
        g_q_i = load_i - b[i]

        alpha = lr_p / np.sqrt(t)
        beta  = lr_q / np.sqrt(t)

        p = np.maximum(0, p + alpha * g_p)
        q[i] = np.maximum(0, q[i] + beta * g_q_i)

        # E. Ghi Log định kỳ
        if t % log_freq == 0:
            term_p = np.sum(p * supply_s)
            term_q = np.sum(q * b)
            term_u = np.sum(budgets * np.log(util + 1e-12))
            obj = term_p + term_q + term_u - np.sum(budgets)

            obj_hist.append(obj)
            real_allocation_hist.append(X.copy())

            if t % 5000 == 0:
                print(f"Iter {t}/{num_iters} | Obj: {obj:.4f}")

    total_time = time.perf_counter() - t0
    print(f"--- Finished in {total_time:.2f}s ---")

    return obj_hist, real_allocation_hist

# ==============================================================================
# 2. MAIN SCRIPT
# ==============================================================================

# --- A. LOAD VALUATIONS ---
FACTOR_DIR = "/content/drive/MyDrive/EV_charging_project/datasets/valuation_new/val_10u_24i"
print(f"--- Loading Data ---")
try:
    val_path = os.path.join(FACTOR_DIR, "valuation_matrix.npy")
    if os.path.exists(val_path):
        valuations = np.load(val_path)
    else:
        U = np.load(os.path.join(FACTOR_DIR, "user_factors.npy"))
        P = np.load(os.path.join(FACTOR_DIR, "item_factors.npy"))
        valuations = U @ P.T
    print(f"✓ Valuations Shape: {valuations.shape}")
except:
    print("⚠️ Dùng dữ liệu ngẫu nhiên (Random Valuations)")
    valuations = np.random.rand(10, 24) + 0.1
valuations /= 10

N_USERS, M_GOODS = valuations.shape
T_SLOTS = 4

--- Loading Data ---
✓ Valuations Shape: (10, 24)


In [28]:
# ==============================================================================
# 3. PARAMETER
# ==============================================================================
np.random.seed(1)

# [1] Budgets
budgets = np.ones(N_USERS) * 10

# [2] Supply
supply_s = np.random.uniform(5, 10, M_GOODS)

# [3] Capacity (Ma trận N x T)
capacity_b = np.random.uniform(1, 4, (N_USERS, T_SLOTS))

# Matrix A
if M_GOODS % T_SLOTS == 0:
    A_matrix = build_A_matrix(M_GOODS, T_SLOTS)
else:
    A_matrix = np.zeros((T_SLOTS, M_GOODS))

print(f"\n--- Params Check ---")
print(f"Total Budget: {np.sum(budgets):.2f}")
print(f"Supply Sum: {np.sum(supply_s):.2f}")
NUM_ITERS = 150000
LOG_FREQ = 10


--- Params Check ---
Total Budget: 100.00
Supply Sum: 170.60


In [33]:
import pickle
import os
import numpy as np

# ==============================================================================
# 4. EXPERIMENT RUNNER (ALL UTILITIES) - UPDATED VERSION
# ==============================================================================

# Cấu hình chung
UTILITY_TYPES = ["CES_8"]
NUM_ITERS = 150000
LOG_FREQ = 10
SAVE_DIR = "/content/drive/MyDrive/EV_charging_project/experiment_results/Fig1_20/12"

# Tạo thư mục nếu chưa tồn tại
if not os.path.exists(SAVE_DIR):
    os.makedirs(SAVE_DIR)
    print(f"📂 Đã tạo thư mục: {SAVE_DIR}")

# Khởi tạo biến Dual (Reset cho mỗi loop để công bằng)
p0_init = np.ones(M_GOODS) * 1.0  # Khởi tạo giá = 1
q0_init = np.zeros((N_USERS, T_SLOTS))

print(f"🚀 Bắt đầu chạy thí nghiệm cho {len(UTILITY_TYPES)} loại hàm Utility...\n")

for u_type in UTILITY_TYPES:
    print(f"==================================================")
    print(f"▶️  PROCESSING: {u_type.upper()}")
    print(f"==================================================")

    # --- BƯỚC 1: TÌM V* VÀ X* (GROUND TRUTH - CENTRALIZED) ---
    try:
        # CẬP NHẬT: Hứng cả 2 giá trị (Optimal Value và Optimal Allocation)
        target_optimal_val, target_optimal_X = solve_centralized_optimal_fair(
            valuations, budgets, supply_s, capacity_b, A_matrix, utility_type=u_type
        )
        print(f"   ★ Target Dual Optimal (V*): {target_optimal_val:.4f}")
        if target_optimal_X is not None:
            print(f"   ★ Optimal Allocation Shape (X*): {target_optimal_X.shape}")

    except Exception as e:
        print(f"   ⚠️ Lỗi Solver Tập trung: {e}")
        target_optimal_val = None
        target_optimal_X = None

    # --- BƯỚC 2: CHẠY SGD (DISTRIBUTED) ---
    current_lr = 0.1

    # CẬP NHẬT: Hứng cả lịch sử Objective và Allocation (X_t)
    # alloc_hist[0] sẽ là X_0
    obj_hist, alloc_hist = one_sample_sgd_fastlog(
        A=A_matrix,
        b=capacity_b,
        supply_s=supply_s,
        q0=q0_init.copy(),      # Copy để không bị dính dữ liệu vòng lặp trước
        valuations=valuations,
        budgets=budgets,
        p0=p0_init.copy(),      # Copy
        lr_p=current_lr,
        lr_q=current_lr,
        num_iters=NUM_ITERS,
        log_freq=LOG_FREQ,
        utility_type=u_type
    )

    # --- BƯỚC 3: LƯU KẾT QUẢ ---
    # Chuẩn hóa tên file: "Linear" -> "linear", "Cobb-Douglas" -> "cobb_douglas"
    safe_name = u_type.lower().replace("-", "_")
    file_name = f"experiment_data_{safe_name}.pkl"
    file_path = os.path.join(SAVE_DIR, file_name)

    # Đóng gói dữ liệu đầy đủ
    result_package = {
        "utility_type": u_type,

        # Dữ liệu từ SGD
        "obj_hist": obj_hist,          # Lịch sử giá trị hàm mục tiêu
        "alloc_hist": alloc_hist,      # Lịch sử phân bổ X (gồm cả X_0)

        # Dữ liệu đối chứng (Ground Truth)
        "optimal_val": target_optimal_val,
        "optimal_X": target_optimal_X, # Ma trận X tối ưu

        # Tham số cấu hình
        "params": {
            "lr": current_lr,
            "iters": NUM_ITERS,
            "log_freq": LOG_FREQ
        }
    }

    with open(file_path, 'wb') as f:
        pickle.dump(result_package, f)

    print(f"   ✅ Đã lưu: {file_name}")
    print(f"   (Final SGD Value: {obj_hist[-1]:.4f})")
    print(f"   (Allocations recorded: {len(alloc_hist)} snapshots)\n")

print("🏁 HOÀN THÀNH TẤT CẢ THÍ NGHIỆM!")

🚀 Bắt đầu chạy thí nghiệm cho 1 loại hàm Utility...

▶️  PROCESSING: CES_8
--- Solving Centralized Fair Problem (CES_8) ---
   ★ Target Dual Optimal (V*): -68.6614
   ★ Optimal Allocation Shape (X*): (10, 24)
--- Start SGD (150000 iterations) | Initial Obj: 11.84 ---
Iter 5000/150000 | Obj: -66.6214
Iter 10000/150000 | Obj: -67.3449
Iter 15000/150000 | Obj: -67.5935
Iter 20000/150000 | Obj: -67.7719
Iter 25000/150000 | Obj: -67.9173
Iter 30000/150000 | Obj: -68.0476
Iter 35000/150000 | Obj: -68.1674
Iter 40000/150000 | Obj: -68.2771
Iter 45000/150000 | Obj: -68.3799
Iter 50000/150000 | Obj: -68.4765
Iter 55000/150000 | Obj: -68.5507
Iter 60000/150000 | Obj: -68.5931
Iter 65000/150000 | Obj: -68.6188
Iter 70000/150000 | Obj: -68.6358
Iter 75000/150000 | Obj: -68.6459
Iter 80000/150000 | Obj: -68.6517
Iter 85000/150000 | Obj: -68.6551
Iter 90000/150000 | Obj: -68.6571
Iter 95000/150000 | Obj: -68.6584
Iter 100000/150000 | Obj: -68.6593
Iter 105000/150000 | Obj: -68.6599
Iter 110000/15000

In [ ]:
import matplotlib.pyplot as plt
import pickle
import os
import numpy as np

# ==============================================================================
# 5. VẼ BIỂU ĐỒ SO SÁNH (SINGLE CHART - OBJECTIVE VALUE)
# ==============================================================================

# Cấu hình đường dẫn
SAVE_DIR = "/content/drive/MyDrive/EV_charging_project/experiment_results/Fig1_20/12"
UTILITY_LIST = ["Linear", "Cobb-Douglas", "Leontief", "CES_8"]

# Cấu hình hiển thị
plt.figure(figsize=(10, 6)) # Kích thước chuẩn cho 1 biểu đồ

colors = {
    "Linear": "#1f77b4",       # Xanh dương
    "Cobb-Douglas": "#2ca02c", # Xanh lá
    "Leontief": "#d62728",      # Đỏ
    "CES_8": "#ff7f0e"         # Cam
}

print("--- Loading & Plotting Data ---")

# Biến tạm để lưu số user cho label trục X
label_n_users = 10

for u_name in UTILITY_LIST:
    # 1. Tạo tên file và load dữ liệu
    safe_name = u_name.lower().replace("-", "_")
    file_path = os.path.join(SAVE_DIR, f"experiment_data_{safe_name}.pkl")

    if not os.path.exists(file_path):
        print(f"⚠️ Không tìm thấy file: {file_path}")
        continue

    with open(file_path, 'rb') as f:
        data = pickle.load(f)

    # 2. Lấy dữ liệu an toàn từ Dictionary
    hist = data["obj_hist"]
    optimal_val = data["optimal_val"]

    # Lấy tham số cấu hình từ file pickle (để code tự động hoá)
    params = data.get("params", {})
    log_freq = params.get("log_freq", 10) # Default là 10 nếu không thấy

    # Tự động xác định N_USERS từ dữ liệu Optimal_X (nếu có)
    if data.get("optimal_X") is not None:
        n_users = data["optimal_X"].shape[0]
    else:
        n_users = 10 # Fallback mặc định

    label_n_users = n_users # Cập nhật cho label trục X

    # 3. Tính toán trục X (Epochs)
    # Epoch = (Số bước lặp) / (Số người dùng)
    steps = np.arange(len(hist)) * log_freq
    epochs = steps / n_users

    c = colors[u_name] # Lấy màu tương ứng

    # 4. Vẽ đường SGD (Nét liền)
    plt.plot(epochs, hist, color=c, linestyle='-', linewidth=2,
             label=f'{u_name} (SGD)')

    # 5. Vẽ đường Optimal (Nét đứt)
    if optimal_val is not None:
        plt.axhline(y=optimal_val, color=c, linestyle='--', linewidth=1.5, alpha=0.7)

        # Ghi chú giá trị (Annotate) ở bên phải biểu đồ
        plt.text(epochs[-1], optimal_val, f'{optimal_val:.1f}',
                 verticalalignment='bottom', horizontalalignment='right',
                 color=c, fontweight='bold', fontsize=9, backgroundcolor='white')

# ==============================================================================
# TRANG TRÍ BIỂU ĐỒ
# ==============================================================================
plt.xlabel(f'Effective Epochs (Iterations / {label_n_users} Users)', fontsize=12)
plt.ylabel('Objective Value', fontsize=12)
plt.title('Convergence Analysis',
          fontsize=13, fontweight='bold')

plt.grid(True, linestyle=':', alpha=0.6)
plt.legend(loc='lower right', frameon=True, shadow=True)

plt.tight_layout()
plt.show()

## 1b

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pickle
import os

# ==============================================================================
# 1. CẤU HÌNH & KHỞI TẠO
# ==============================================================================
SAVE_DIR = "/content/drive/MyDrive/EV_charging_project/experiment_results/Fig1_20/12"
UTILITY_TYPES = ["Linear", "Cobb-Douglas", "Leontief", "CES_8"]
WINDOW_SIZE = 50  # Tăng lên 50 để đường vẽ mượt hơn (SGD thường rất rung)

# Cấu hình màu và kiểu đường (THÊM CES_8 VÀO ĐÂY)
STYLES = {
    "Linear":       {"color": "#1f77b4", "linestyle": "-",  "label": "Linear"},        # Xanh dương
    "Cobb-Douglas": {"color": "#2ca02c", "linestyle": "--", "label": "Cobb-Douglas"},  # Xanh lá
    "Leontief":     {"color": "#d62728", "linestyle": "-.", "label": "Leontief"},      # Đỏ
    "CES_8":        {"color": "#9467bd", "linestyle": ":",  "label": "CES (m=8)"}      # Tím
}

plt.figure(figsize=(10, 6))
captured_n_users = 10

# ==============================================================================
# 2. VÒNG LẶP XỬ LÝ & VẼ
# ==============================================================================
print(f"--- Processing Convergence Plot (Window Size: {WINDOW_SIZE}) ---")

for u_type in UTILITY_TYPES:
    # 1. Load file
    safe_name = u_type.lower().replace("-", "_")
    file_path = os.path.join(SAVE_DIR, f"experiment_data_{safe_name}.pkl")

    # --- SỬA LỖI 1: Lấy style tương ứng ---
    style = STYLES.get(u_type, {"color": "black", "linestyle": "-", "label": u_type})

    if not os.path.exists(file_path):
        print(f"⚠️  [MISSING] Không tìm thấy file: {file_path}")
        continue

    try:
        with open(file_path, "rb") as f:
            data = pickle.load(f)
    except Exception as e:
        print(f"❌ [ERROR] Lỗi đọc file {u_type}: {e}")
        continue

    # 2. Lấy dữ liệu
    X_star = data.get("optimal_X")
    alloc_hist = data.get("alloc_hist")
    params = data.get("params", {})
    log_freq = params.get("log_freq", 10)


    # 3. Kiểm tra dữ liệu
    if X_star is None:
        print(f"⏭️  [SKIP] {u_type}: Không có nghiệm tối ưu trung tâm (X*) để so sánh.")
        # Lưu ý: CES_8 thường sẽ rơi vào case này nếu Solver tập trung ko chạy được
        continue

    if alloc_hist is None or len(alloc_hist) == 0:
        print(f"⏭️  [SKIP] {u_type}: Không có lịch sử Allocation.")
        continue

    print(f"✅ [PLOTTING] {u_type}...")

    # Cập nhật số user để chia Epoch cho đúng
    n_users_current = X_star.shape[0]
    captured_n_users = n_users_current

    # --- TÍNH TOÁN NORMALIZED RESIDUALS ---
    # Residual = ||X_k - X*||_F / ||X_0 - X*||_F

    # Mẫu số: Sai số ban đầu
    X_0 = alloc_hist[0]
    denom = np.linalg.norm(X_0 - X_star, 'fro')

    residuals = []
    for X_k in alloc_hist:
        num = np.linalg.norm(X_k - X_star, 'fro')
        residuals.append(num / denom)

    residuals = np.array(residuals)

    # --- XỬ LÝ SMOOTHING (SMA) ---
    if len(residuals) >= WINDOW_SIZE:
        residuals_smoothed = np.convolve(
            residuals,
            np.ones(WINDOW_SIZE)/WINDOW_SIZE,
            mode='valid'
        )
        # Tạo trục X (Epochs) tương ứng
        # Do mode='valid', mảng ngắn đi (WINDOW_SIZE - 1) phần tử
        raw_iterations = np.arange(len(residuals)) * log_freq

        # Cắt trục X để khớp độ dài dữ liệu đã smooth
        # Lấy từ phần tử thứ (WINDOW_SIZE - 1) trở đi
        valid_iterations = raw_iterations[WINDOW_SIZE - 1:]
        epochs_to_plot = valid_iterations / n_users_current
        data_to_plot = residuals_smoothed
    else:
        # Fallback nếu dữ liệu quá ngắn
        raw_iterations = np.arange(len(residuals)) * log_freq
        epochs_to_plot = raw_iterations / n_users_current
        data_to_plot = residuals

    # --- VẼ ---
    plt.semilogy(epochs_to_plot, data_to_plot,
                 color=style["color"],
                 linestyle=style["linestyle"],
                 linewidth=2.0,
                 alpha=0.9,
                 label=style["label"])

# ==============================================================================
# 3. TRANG TRÍ & HOÀN THIỆN CHART
# ==============================================================================
plt.xlabel(f'Effective Epochs (Iterations / {captured_n_users} Users)', fontsize=12)
plt.ylabel(r'Normalized Residual $\frac{\|\mathbf{X}^{(k)} - \mathbf{X}^*\|_F}{\|\mathbf{X}^{(0)} - \mathbf{X}^*\|_F}$', fontsize=14)
plt.title(f'Convergence of Allocation (Smoothed Window={WINDOW_SIZE})', fontsize=14, fontweight='bold')

plt.grid(True, which="major", linestyle='-', alpha=0.5)
plt.grid(True, which="minor", linestyle=':', alpha=0.2)
plt.legend(fontsize=11, loc='upper right', framealpha=0.9, shadow=True)

plt.tight_layout()
plt.show()